In [1]:
# required libraries
import os
import pickle
import torch
import torch.nn as nn

import numpy as np
import pandas as pd
import math

import data.processor as preprocess

import mlmodel.mlp as model_mlp
import mlmodel.softdt as model_softdt
import mlmodel.tab_transformer as model_tabtrans
import mlmodel.simple_vae as model_simple_vae

processed_data_path = os.path.join(os.getcwd(), 'data/preprocessed/')
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')

from attack.vae_attack import VAEAttack
import attack.run_gridsearch as run_gridsearch

# Remove the Future warning from pytorch
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
LOAD_EXIST = True

In [3]:
def pick_true_prediction(model, data):
    X_test_tensor, y_test_tensor = data

    with torch.no_grad():
        model.eval()
        test_outputs = model(X_test_tensor.to(device))
        predicted = torch.argmax(test_outputs, dim=1).float()
        prediction = (predicted == y_test_tensor.to(device))

        # Create tensors to hold true predictions' indices, inputs, and targets
        indices_of_true_predictions = []
        X_of_true_predictions = []
        y_of_true_predictions = []

        for i in range(len(prediction)):
            if prediction[i]:
                indices_of_true_predictions.append(i)
                X_of_true_predictions.append(X_test_tensor[i])
                y_of_true_predictions.append(y_test_tensor[i])

        return (
            torch.tensor(indices_of_true_predictions),
            torch.stack(X_of_true_predictions),
            torch.stack(y_of_true_predictions),
        )


In [4]:
parameter_grid_baseline = {
    '_lambda': [1], # Regularization parameter, 0.01, 0.03, 0.1, 0.3, 0.5, 0.7, 1 # 0.6, 0.7, 0.8, 0.9, 1
    'optimizer': ['adam'],  # Using Adam optimizer
    'lr': [0.1],  # Learning rates to try, 0.01, 0.03, 0.1, 0.15, 0.2, 0.25, 0.3
    'max_iter': [300],  # Maximum iterations to try
    'sample_num': [500],  # Number of samples to generate
    'kappa': [0],  # Kappa value
}

In [5]:
parameter_grid_search = {
    '_lambda': [0.5, 0.6, 0.7, 0.8, 0.9, 1], # Regularization parameter, # 0.6, 0.7, 0.8, 0.9, 1
    'optimizer': ['adam'],  # Using Adam optimizer
    'lr': [0.01, 0.03, 0.1, 0.15, 0.2, 0.25, 0.3],  # Learning rates to try
    'max_iter': [300],  # Maximum iterations to try
    'sample_num': [500],  # Number of samples to generate
    'kappa': [0],  # Kappa value
}



### Adult

In [6]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/adult')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([21112, 12]), y_train: torch.Size([21112])
X_val: torch.Size([3017, 12]), y_val: torch.Size([3017])
X_test: torch.Size([6033, 12]), y_test: torch.Size([6033])
The num of embedding dim for Transformer is: 10
The num of categorical features is: 7
The num of continues features is: 5
The num of total features is: 12
The num of binary features is: 1
Combined embedding dims: [(16, 4), (7, 3), (7, 3), (14, 4), (6, 3), (5, 3), (41, 7)]


Load VAE model

In [7]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "adult",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 16,
    "hidden_layers": [128, 64],
    "kl_weight": 1e-3,
    "classification_weight": 1,
}



simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)


simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/adult_0.001_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(16, 4)
    (1-2): 2 x Embedding(7, 3)
    (3): Embedding(14, 4)
    (4): Embedding(6, 3)
    (5): Embedding(5, 3)
    (6): Embedding(41, 7)
  )
  (encoder): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=64, out_features=16, bias=True)
  (_log_var_enc): Linear(in_features=64, out_features=16, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0

Load ml model

In [8]:
import mlmodel.mlp as model_mlp

mlp_config = {
    "model": "MLP",
    "dataset": "adult",
    "epochs": 150,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-4, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_adult.pt


In [9]:
import mlmodel.softdt as model_softdt

softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "adult",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num, 
                                       output_dim=2, 
                                       depth=5, 
                                       lamda=0.01, 
                                       num_categorical=categorical_num,
                                       embedding_dims=embedding_dims,
                                       device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_adult.pt


In [10]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "adult",
    "epochs": 10,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                      num_continuous=continues_num,
                                      dim=trans_dim,
                                      depth=3,
                                      heads=4,
                                      dim_head=16,
                                      dim_out=2,
                                      ff_dropout=0.2,
                                      attn_dropout=0.2,
                                      )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-3, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_adult.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [11]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="adult",                         # Dataset name
        model=model_str,                               # Model name
        attack="baseline",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 5151
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.51,
        "mean_l2_distance": 0.6220138669013977,
        "mean_linf_distance": 0.3022327721118927,
        "perturbed_features_latent": 14.431372549019608,
        "perturbed_features_input": 4.662745098039216,
        "mean_confidence_successful": 0.5615181827296813,
        "mean_confidence_unsuccessful": 0.1944775829509813,
        "min_confidence_unsuccessful": 0.0

Run hyperparameter search

In [12]:
for model_str in ["MLP"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_search['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="adult",                         # Dataset name
        model=model_str,                               # Model name
        attack="baseline",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_search,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 5151
------------------------------------------------------------
Running combination 1/42

Testing parameters: {'_lambda': 0.5, 'optimizer': 'adam', 'lr': 0.01, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 0.5, 'optimizer': 'adam', 'lr': 0.01, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 0.5,
        "optimizer": "adam",
        "lr": 0.01,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.212,
        "mean_l2_distance": 0.20227394998073578,
        "mean_linf_distance": 0.10137373208999634,
        "perturbed_features_latent": 12.226415094339623,
        "perturbed_features_input": 4.245283018867925,
        "mean_confidence_successful": 0.5480000267227022,
        "mean_confidence_unsuccessful": 0.17100208891829863,
        "min_confidence_unsu

### phishing_url

In [13]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/phishing_url')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([8001, 86]), y_train: torch.Size([8001])
X_val: torch.Size([1143, 86]), y_val: torch.Size([1143])
X_test: torch.Size([2286, 86]), y_test: torch.Size([2286])
The num of embedding dim for Transformer is: 2
The num of categorical features is: 1
The num of continues features is: 85
The num of total features is: 86
The num of binary features is: 28
Combined embedding dims: [(3, 2)]


In [14]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "phishing_url",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 16,
    "hidden_layers": [128, 64],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/phishing_url_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(3, 2)
  )
  (encoder): Sequential(
    (0): Linear(in_features=87, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=64, out_features=16, bias=True)
  (_log_var_enc): Linear(in_features=64, out_features=16, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=16, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_f

Load ml model

In [15]:

mlp_config = {
    "model": "MLP",
    "dataset": "phishing_url",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-4, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_phishing_url.pt


In [16]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "phishing_url",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_phishing_url.pt


In [17]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "phishing_url",
    "epochs": 50,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_phishing_url.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [18]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="phishing_url",                         # Dataset name
        model=model_str,                               # Model name
        attack="baseline",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 2198
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.98,
        "mean_l2_distance": 0.9132367968559265,
        "mean_linf_distance": 0.493863046169281,
        "perturbed_features_latent": 15.738775510204082,
        "perturbed_features_input": 59.13061224489796,
        "mean_confidence_successful": 0.6488929253869823,
        "mean_confidence_unsuccessful": 0.00812779664993286,
        "min_confidence_unsuccessful": 4.4

### PenDigit

In [19]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/pendigit')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([7693, 16]), y_train: torch.Size([7693])
X_val: torch.Size([1100, 16]), y_val: torch.Size([1100])
X_test: torch.Size([2199, 16]), y_test: torch.Size([2199])
The num of embedding dim for Transformer is: 0
The num of categorical features is: 0
The num of continues features is: 16
The num of total features is: 16
The num of binary features is: 0
Combined embedding dims: []


Load VAE model

In [20]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "pendigit",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 8,
    "hidden_layers": [64, 32],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}



simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
    num_classes=10
)


simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/pendigit_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList()
  (encoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=32, out_features=8, bias=True)
  (_log_var_enc): Linear(in_features=32, out_features=8, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=8, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=64, bias=True)
      (3): Re

Load ml model

In [21]:
import mlmodel.mlp as model_mlp

mlp_config = {
    "model": "MLP",
    "dataset": "pendigit",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=10,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_pendigit.pt


In [22]:
import mlmodel.softdt as model_softdt

softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "pendigit",
    "epochs": 250,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num, 
                                       output_dim=10, 
                                       depth=5, 
                                       lamda=0.01, 
                                       num_categorical=categorical_num,
                                       embedding_dims=embedding_dims,
                                       device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_pendigit.pt


In [23]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "pendigit",
    "epochs": 250,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=[],
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=10,
                                      )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')


Model loaded from models/TabTransformer_pendigit.pt


/opt/homebrew/Caskroom/miniforge/base/envs/VAE-Attack/lib/python3.11/site-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Apply attacks - MLP + SoftDT + Tab Transfomer

In [24]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="pendigit",                         # Dataset name
        model=model_str,                               # Model name
        attack="baseline",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 2154
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.908,
        "mean_l2_distance": 1.122784972190857,
        "mean_linf_distance": 0.693932056427002,
        "perturbed_features_latent": 7.927312775330397,
        "perturbed_features_input": 16.0,
        "mean_confidence_successful": 0.5938858502079827,
        "mean_confidence_unsuccessful": 0.035403629359991654,
        "min_confidence_unsuccessful": 9.69171524047851

### Mushroom

In [25]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/mushroom')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([3950, 22]), y_train: torch.Size([3950])
X_val: torch.Size([565, 22]), y_val: torch.Size([565])
X_test: torch.Size([1129, 22]), y_test: torch.Size([1129])
The num of embedding dim for Transformer is: 10
The num of categorical features is: 18
The num of continues features is: 4
The num of total features is: 22
The num of binary features is: 4
Combined embedding dims: [(6, 3), (4, 2), (8, 3), (7, 3), (2, 2), (2, 2), (9, 3), (4, 2), (4, 2), (4, 2), (7, 3), (7, 3), (2, 2), (3, 2), (4, 2), (6, 3), (6, 3), (6, 3)]


In [26]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "mushroom",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 10,
    "hidden_layers": [64, 32],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/mushroom_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(6, 3)
    (1): Embedding(4, 2)
    (2): Embedding(8, 3)
    (3): Embedding(7, 3)
    (4-5): 2 x Embedding(2, 2)
    (6): Embedding(9, 3)
    (7-9): 3 x Embedding(4, 2)
    (10-11): 2 x Embedding(7, 3)
    (12): Embedding(2, 2)
    (13): Embedding(3, 2)
    (14): Embedding(4, 2)
    (15-17): 3 x Embedding(6, 3)
  )
  (encoder): Sequential(
    (0): Linear(in_features=49, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=32, out_features=10, bias=True)
  (_log_var_enc): Linear(in_features=32, out_features=10, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in

Load ml model

In [27]:

mlp_config = {
    "model": "MLP",
    "dataset": "mushroom",
    "epochs": 50,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_mushroom.pt


In [28]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "mushroom",
    "epochs": 75,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_mushroom.pt


In [29]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "mushroom",
    "epochs": 10,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=2,
                                         ff_dropout=0.2,
                                         attn_dropout=0.2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_mushroom.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [30]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="mushroom",                         # Dataset name
        model=model_str,                               # Model name
        attack="baseline",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 1129
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.0,
        "mean_l2_distance": 0.0,
        "mean_linf_distance": 0.0,
        "perturbed_features_latent": 0.0,
        "perturbed_features_input": 0.0,
        "mean_confidence_successful": 0.0,
        "mean_confidence_unsuccessful": 5.3428173065185545e-05,
        "min_confidence_unsuccessful": 0.0,
        "max_confidence_unsuccessful": 0.0015467405319213867,
       

### german_credit

In [31]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/german_credit')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([700, 19]), y_train: torch.Size([700])
X_val: torch.Size([100, 19]), y_val: torch.Size([100])
X_test: torch.Size([200, 19]), y_test: torch.Size([200])
The num of embedding dim for Transformer is: 8
The num of categorical features is: 11
The num of continues features is: 8
The num of total features is: 19
The num of binary features is: 2
Combined embedding dims: [(4, 2), (5, 3), (10, 4), (5, 3), (5, 3), (4, 2), (3, 2), (4, 2), (3, 2), (3, 2), (4, 2)]


In [32]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "german_credit",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 8,
    "hidden_layers": [64, 32],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/german_credit_0.01_100_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(4, 2)
    (1): Embedding(5, 3)
    (2): Embedding(10, 4)
    (3-4): 2 x Embedding(5, 3)
    (5): Embedding(4, 2)
    (6): Embedding(3, 2)
    (7): Embedding(4, 2)
    (8-9): 2 x Embedding(3, 2)
    (10): Embedding(4, 2)
  )
  (encoder): Sequential(
    (0): Linear(in_features=35, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=32, out_features=8, bias=True)
  (_log_var_enc): Linear(in_features=32, out_features=8, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_f

Load ml model

In [33]:

mlp_config = {
    "model": "MLP",
    "dataset": "german_credit",
    "epochs": 50,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.1
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_german_credit.pt


In [34]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "german_credit",
    "epochs": 30,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-2, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_german_credit.pt


In [35]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "german_credit",
    "epochs": 10,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=2,
                                         heads=3,
                                         dim_head=5,
                                         dim_out=2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-2, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_german_credit.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [36]:

for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="german_credit",                         # Dataset name
        model=model_str,                               # Model name
        attack="baseline",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
        override_sample_num=len(sampled_prediction_y_tensor)
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 149
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.09395973154362416,
        "mean_l2_distance": 0.0014686895301565528,
        "mean_linf_distance": 0.0010295595275238156,
        "perturbed_features_latent": 1.1428571428571428,
        "perturbed_features_input": 9.642857142857142,
        "mean_confidence_successful": 0.5397877033267703,
        "mean_confidence_unsuccessful": 0.21770363648732502,
        "min_confiden

### electricity

In [37]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/electricity')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([31717, 8]), y_train: torch.Size([31717])
X_val: torch.Size([4532, 8]), y_val: torch.Size([4532])
X_test: torch.Size([9063, 8]), y_test: torch.Size([9063])
The num of embedding dim for Transformer is: 3
The num of categorical features is: 1
The num of continues features is: 7
The num of total features is: 8
The num of binary features is: 0
Combined embedding dims: [(7, 3)]


In [38]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "electricity",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 8,
    "hidden_layers": [16],
    "kl_weight": 1e-4,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/electricity_0.0001_100_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(7, 3)
  )
  (encoder): Sequential(
    (0): Linear(in_features=10, out_features=16, bias=True)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=16, out_features=8, bias=True)
  (_log_var_enc): Linear(in_features=16, out_features=8, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=8, out_features=16, bias=True)
      (1): ReLU()
    )
    (1): Linear(in_features=16, out_features=8, bias=True)
    (2): Sigmoid()
  )
  (cat_decoder_layers): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=16, out_features=16, bias=True)
      (1): ReLU()
      (2): Linear(in_features=16, out_features=7, bias=True)
    )
  )
  (num_decoder): Sequential(
  

Load ml model

In [39]:

mlp_config = {
    "model": "MLP",
    "dataset": "electricity",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.1
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_electricity.pt


In [40]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "electricity",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-2, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_electricity.pt


In [41]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "electricity",
    "epochs": 100,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=2,
                                         heads=3,
                                         dim_head=4,
                                         dim_out=2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-3, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_electricity.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [42]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="electricity",                         # Dataset name
        model=model_str,                               # Model name
        attack="baseline",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 7351
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.802,
        "mean_l2_distance": 0.5009191036224365,
        "mean_linf_distance": 0.3418301045894623,
        "perturbed_features_latent": 7.12219451371571,
        "perturbed_features_input": 7.022443890274314,
        "mean_confidence_successful": 0.5198255517238988,
        "mean_confidence_unsuccessful": 0.2774955714591826,
        "min_confidence_unsuccessful": 1.78

### covertype

In [43]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/covertype')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([406707, 54]), y_train: torch.Size([406707])
X_val: torch.Size([58102, 54]), y_val: torch.Size([58102])
X_test: torch.Size([116203, 54]), y_test: torch.Size([116203])
The num of embedding dim for Transformer is: 0
The num of categorical features is: 0
The num of continues features is: 54
The num of total features is: 54
The num of binary features is: 44
Combined embedding dims: []


In [44]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "covertype",
    "epochs": 50,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 16,
    "hidden_layers": [128, 64],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
    num_classes=7,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/covertype_0.01_50_1.pt


SimpleVAE(
  (embedding_layers): ModuleList()
  (encoder): Sequential(
    (0): Linear(in_features=54, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=64, out_features=16, bias=True)
  (_log_var_enc): Linear(in_features=64, out_features=16, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=16, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=128, bias=True)
    

Load ml model

In [45]:

mlp_config = {
    "model": "MLP",
    "dataset": "covertype",
    "epochs": 50,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=7,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.1
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_covertype.pt


In [46]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "covertype",
    "epochs": 50,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=7,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_covertype.pt


In [47]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "covertype",
    "epochs": 30,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=4,
                                         heads=5,
                                         dim_head=10,
                                         dim_out=7,
                                         attn_dropout=0.1,
                                         ff_dropout=0.1,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-2, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_covertype.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [48]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]:

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    print(sampled_true_prediction_X_tensor.shape)
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="covertype",                         # Dataset name
        model=model_str,                               # Model name
        attack="baseline",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
        override_sample_num=len(sampled_prediction_y_tensor)
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 97135
torch.Size([497, 54])
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Successfully loaded existing results for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0}
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0
    },
    "metrics": {
        "success_rate": 0.8873239436619719,
        "mean_l2_distance": 0.35558903217315674,
        "mean_linf_distance": 0.20858058333396912,
        "perturbed_features_latent": 11.71655328798186,
        "perturbed_features_input": 10.065759637188208,
        "mean_confidence_successful": 0.5861813286802459,
        "mean_confidence_unsuccessful": 0.3427207890365805,
   